<a href="https://colab.research.google.com/github/arjunnamburi/ml_death_overs_ipl/blob/main/archive_notebooks/ML_Project_IPL_3.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
import hashlib
import numpy as np
import pandas as pd
from google.colab import drive

drive.mount('/content/drive')
DATA = '/content/drive/MyDrive/IPL PROJECT ALL DOCS/data/'   # every file is read from and written to this folder


In [ ]:
df = pd.read_csv(DATA + 'prog2_renamed.csv')
pd.set_option('display.max_columns', None)

In [ ]:
filt = (df['innings'] > 2)
rows_to_drop = df[df['innings'] > 2].index
df.drop(index=rows_to_drop, inplace=True)

In [ ]:
inn = (df[df.innings.isin([1, 2])]
       .groupby(['match_id', 'innings']).agg(
            max_over=('over', 'max'),
            runs=('ball_runs_total', 'sum'),
            wkts=('c_team_wickets', 'max'),
            target=('runs_target', 'first')
            ).unstack('innings'))

In [ ]:

TARGET_IS_WINNING_SCORE = True   # True if runs_target == innings-1 total + 1 in a normal match
FULL_BALLS = 120                 # legal balls in a full innings

# Regular innings only (super overs already dropped)
base = df[df['innings'].isin([1, 2])]

inn = (base.groupby(['match_id', 'innings'])
       .agg(legal_balls=('valid_ball', 'sum'),
            runs=('ball_runs_total', 'sum'),
            wkts=('player_out', 'count'),        # total dismissals (NaN on non-wicket balls)
            target=('runs_target', 'first'))
       .unstack('innings'))

win_score2 = inn[('target', 2)] if TARGET_IS_WINNING_SCORE else inn[('target', 2)] + 1

all_out1  = inn[('wkts', 1)] >= 10
all_out2  = inn[('wkts', 2)] >= 10
chase_won = inn[('runs', 2)] >= win_score2

no_inn2 = inn[('legal_balls', 2)].isna()                      # match ended after innings 1
short1  = (inn[('legal_balls', 1)] < FULL_BALLS) & ~all_out1
short2  = ((inn[('legal_balls', 2)] < FULL_BALLS) & ~all_out2
           & ~chase_won & ~no_inn2)
revised = (~no_inn2) & (win_score2 != inn[('runs', 1)] + 1)   # DLS / revised target

flag = short1 | short2 | revised | no_inn2

reason = (short1.map({True: 'short1;', False: ''})
          + short2.map({True: 'short2;', False: ''})
          + revised.map({True: 'revised;', False: ''})
          + no_inn2.map({True: 'no_inn2;', False: ''}))

flagged = inn[flag].assign(reason=reason[flag])
print(f"{len(flagged)} flagged of {len(inn)} matches")
print(flagged[[('legal_balls', 1), ('legal_balls', 2),
               ('runs', 1), ('runs', 2),
               ('wkts', 1), ('wkts', 2),
               ('target', 2), ('reason', '')]])

# ---- Hand-review step: edit these after looking at the printout ----
false_positives = []   # match_ids flagged by mistake, to unflag
missed          = []   # match_ids you found by hand that the code missed

rain_ids = (set(flagged.index) - set(false_positives)) | set(missed)

# ---- Create the column (match-level, so both innings rows get the same value) ----
df['rain_affected'] = df['match_id'].isin(rain_ids).astype(int)

# Save the list for your report
pd.Series(sorted(rain_ids), name='match_id').to_csv(DATA + 'rain_affected_matches.csv', index=False)
print(df.groupby('rain_affected')['match_id'].nunique())

In [ ]:
df.to_csv(DATA + 'prog3.csv', index=False)

In [ ]:
print(df.shape)   # expect (295553, ...)
print('prog3.csv md5:', hashlib.md5(open(DATA + 'prog3.csv', 'rb').read()).hexdigest())
# my Oct 6 prog3.csv (the one features.ipynb reads): 4f7dd268da911c286cf3fc0fad7ec8e2
